# Analytics Logic

This notebook builds analytics logic for the airport operations platform: fuel cost savings, delay penalty avoidance, resource cost efficiency, turnaround trends, delay causes, delay pattern heatmaps, utilization history, model improvement, and backend-ready analytics payloads.

## 1. Setup

In [ ]:
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

now = datetime(2026, 4, 27, 9, 0)

## 2. Analytics Configuration

In [ ]:
AIRLINES = ['Air India', 'IndiGo', 'SpiceJet', 'Vistara', 'Akasa', 'AirAsia']
DELAY_CAUSES = ['Gate conflict', 'Fueling overlap', 'Crew positioning', 'Baggage delay', 'ATC slot miss', 'Weather', 'Maintenance', 'Runway queue']
RESOURCE_TYPES = ['Crew', 'Equipment', 'Gates', 'Runways', 'Baggage Belts', 'GPU']
FUEL_PRICE_PER_LITER = 92.5
BASELINE_TAXI_FUEL_LPM = 42
PENALTY_PER_MIN = {'Air India': 520, 'IndiGo': 460, 'SpiceJet': 390, 'Vistara': 540, 'Akasa': 410, 'AirAsia': 360}

## 3. Generate Historical Analytics Dataset

In [ ]:
def generate_analytics_history(days=90, flights_per_day=58):
    rows = []
    start = (now - timedelta(days=days)).replace(hour=0, minute=0, second=0, microsecond=0)
    for day in range(days):
        current_date = start + timedelta(days=day)
        weekday = current_date.weekday()
        for f in range(flights_per_day + random.randint(-8, 12)):
            hour = random.choices(range(24), weights=[2,1,1,1,2,4,8,12,11,8,7,6,6,7,8,8,10,13,14,12,8,5,3,2])[0]
            airline = random.choice(AIRLINES)
            peak = hour in [7, 8, 9, 17, 18, 19]
            ai_optimized = random.random() < (0.42 + min(day / days, 1) * 0.38)
            scheduled_tat = random.choice([35, 42, 48, 55, 75, 90])
            base_delay = np.random.gamma(2.0, 7.5) + (10 if peak else 0) + (6 if weekday == 4 else 0)
            if ai_optimized:
                avoided_delay = base_delay * random.uniform(0.22, 0.58)
            else:
                avoided_delay = 0
            actual_delay = max(0, base_delay - avoided_delay + np.random.normal(0, 4))
            baseline_taxi_min = np.clip(np.random.normal(16 + (4 if peak else 0), 5), 6, 35)
            optimized_taxi_min = max(4, baseline_taxi_min - (random.uniform(1.5, 7.0) if ai_optimized else random.uniform(-1, 1)))
            fuel_saved_liters = max(0, (baseline_taxi_min - optimized_taxi_min) * BASELINE_TAXI_FUEL_LPM)
            penalty_avoided = avoided_delay * PENALTY_PER_MIN[airline]
            resource_cost = random.uniform(18000, 90000) * (1.08 if not ai_optimized else 0.94)
            passenger_count = random.randint(55, 410)
            turnaround_actual = scheduled_tat + actual_delay * 0.42 + np.random.normal(0, 5)
            cause = random.choices(DELAY_CAUSES, weights=[18, 15, 15, 14, 10, 10, 8, 10])[0]

            rows.append({
                'date': current_date.date(),
                'timestamp': current_date + timedelta(hours=hour, minutes=random.randint(0, 59)),
                'weekday': weekday,
                'hour': hour,
                'flight_id': f'{random.choice(["AI", "6E", "SG", "UK", "QP"])}{random.randint(100, 999)}',
                'airline': airline,
                'passenger_count': passenger_count,
                'scheduled_turnaround_min': scheduled_tat,
                'actual_turnaround_min': round(turnaround_actual, 1),
                'delay_minutes': round(actual_delay, 1),
                'avoided_delay_minutes': round(avoided_delay, 1),
                'delay_cause': cause,
                'baseline_taxi_min': round(baseline_taxi_min, 1),
                'optimized_taxi_min': round(optimized_taxi_min, 1),
                'fuel_saved_liters': round(fuel_saved_liters, 1),
                'fuel_cost_saved': round(fuel_saved_liters * FUEL_PRICE_PER_LITER, 1),
                'penalty_avoided': round(penalty_avoided, 1),
                'resource_cost': round(resource_cost, 1),
                'ai_optimized': ai_optimized,
                'model_version': f'v{1 + day // 22}.{day % 22}',
            })
    return pd.DataFrame(rows)


history_df = generate_analytics_history()
history_df.head(10)

## 4. KPI Calculations

In [ ]:
def period_filter(df, days):
    cutoff = now.date() - timedelta(days=days)
    return df[df['date'] >= cutoff]


def build_analytics_kpis(df):
    total_fuel_saved = df['fuel_cost_saved'].sum()
    penalty_avoided = df['penalty_avoided'].sum()
    avg_turnaround = df['actual_turnaround_min'].mean()
    baseline_turnaround = df['scheduled_turnaround_min'].mean() + df['delay_minutes'].mean() * 0.55
    turnaround_improvement = baseline_turnaround - avg_turnaround
    ai_coverage = df['ai_optimized'].mean() * 100
    total_delay_avoided = df['avoided_delay_minutes'].sum()
    avg_resource_cost_per_flight = df['resource_cost'].mean()
    cost_per_passenger = df['resource_cost'].sum() / max(df['passenger_count'].sum(), 1)
    on_time_rate = (df['delay_minutes'] <= 5).mean() * 100

    return pd.DataFrame([{
        'flights_analyzed': len(df),
        'fuel_cost_saved': int(total_fuel_saved),
        'delay_penalty_avoided': int(penalty_avoided),
        'avg_turnaround_min': round(avg_turnaround, 1),
        'turnaround_improvement_min': round(turnaround_improvement, 1),
        'delay_minutes_avoided': int(total_delay_avoided),
        'ai_coverage_pct': round(ai_coverage, 1),
        'avg_resource_cost_per_flight': int(avg_resource_cost_per_flight),
        'resource_cost_per_passenger': round(cost_per_passenger, 1),
        'on_time_rate_pct': round(on_time_rate, 1),
    }])


current_7d = period_filter(history_df, 7)
previous_7d = history_df[(history_df['date'] < now.date() - timedelta(days=7)) & (history_df['date'] >= now.date() - timedelta(days=14))]
kpi_df = build_analytics_kpis(current_7d)
previous_kpi_df = build_analytics_kpis(previous_7d)
kpi_df

## 5. Trend Analytics

In [ ]:
def build_daily_trends(df):
    daily = df.groupby('date').agg(
        flights=('flight_id', 'count'),
        avg_turnaround_min=('actual_turnaround_min', 'mean'),
        avg_delay_min=('delay_minutes', 'mean'),
        fuel_cost_saved=('fuel_cost_saved', 'sum'),
        penalty_avoided=('penalty_avoided', 'sum'),
        delay_minutes_avoided=('avoided_delay_minutes', 'sum'),
        ai_coverage_pct=('ai_optimized', lambda x: x.mean() * 100),
        on_time_rate_pct=('delay_minutes', lambda x: (x <= 5).mean() * 100),
    ).reset_index()
    numeric = daily.select_dtypes(include=[np.number]).columns
    daily[numeric] = daily[numeric].round(1)
    return daily


daily_trends_df = build_daily_trends(history_df)
daily_trends_df.tail(14)

## 6. Delay Cause Analytics

In [ ]:
def build_delay_cause_table(df):
    cause = df.groupby('delay_cause').agg(
        flights=('flight_id', 'count'),
        avg_delay_min=('delay_minutes', 'mean'),
        total_delay_min=('delay_minutes', 'sum'),
        avoided_delay_min=('avoided_delay_minutes', 'sum'),
        penalty_avoided=('penalty_avoided', 'sum'),
    ).reset_index()
    cause['share_pct'] = (cause['flights'] / cause['flights'].sum() * 100).round(1)
    cause['avg_delay_min'] = cause['avg_delay_min'].round(1)
    cause['total_delay_min'] = cause['total_delay_min'].round(1)
    cause['avoided_delay_min'] = cause['avoided_delay_min'].round(1)
    cause['penalty_avoided'] = cause['penalty_avoided'].round(0).astype(int)
    return cause.sort_values('total_delay_min', ascending=False).reset_index(drop=True)


delay_causes_df = build_delay_cause_table(current_7d)
delay_causes_df

## 7. Delay Pattern Heatmap

In [ ]:
def build_delay_heatmap(df):
    heat = df[df['delay_minutes'] > 5].groupby(['weekday', 'hour']).size().reset_index(name='delay_count')
    matrix = heat.pivot(index='weekday', columns='hour', values='delay_count').reindex(index=range(7), columns=range(24), fill_value=0).fillna(0).astype(int)
    matrix.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    return matrix


delay_heatmap_df = build_delay_heatmap(current_7d)
delay_heatmap_df.iloc[:, 6:22]

## 8. Resource Utilization History

In [ ]:
def generate_resource_utilization(daily_trends):
    rows = []
    for _, day in daily_trends.iterrows():
        peak_factor = min(day['flights'] / 65, 1.4)
        for resource in RESOURCE_TYPES:
            base = {'Crew': 76, 'Equipment': 73, 'Gates': 82, 'Runways': 65, 'Baggage Belts': 70, 'GPU': 68}[resource]
            utilization = np.clip(np.random.normal(base * peak_factor, 8), 20, 100)
            idle_hours = max(0, (100 - utilization) / 100 * random.uniform(20, 95))
            rows.append({
                'date': day['date'],
                'resource_type': resource,
                'utilization_pct': round(utilization, 1),
                'idle_hours': round(idle_hours, 1),
                'overload_flag': utilization >= 92,
            })
    return pd.DataFrame(rows)


resource_history_df = generate_resource_utilization(daily_trends_df)
resource_summary_df = resource_history_df[resource_history_df['date'].isin(current_7d['date'].unique())].groupby('resource_type').agg(
    avg_utilization_pct=('utilization_pct', 'mean'),
    max_utilization_pct=('utilization_pct', 'max'),
    total_idle_hours=('idle_hours', 'sum'),
    overload_days=('overload_flag', 'sum'),
).reset_index()
resource_summary_df[['avg_utilization_pct', 'max_utilization_pct', 'total_idle_hours']] = resource_summary_df[['avg_utilization_pct', 'max_utilization_pct', 'total_idle_hours']].round(1)
resource_summary_df

## 9. Model Improvement Log

In [ ]:
def build_model_improvement_log(df):
    weekly = df.copy()
    weekly['week'] = pd.to_datetime(weekly['date']).dt.isocalendar().week.astype(int)
    model_log = weekly.groupby('week').agg(
        flights=('flight_id', 'count'),
        ai_coverage_pct=('ai_optimized', lambda x: x.mean() * 100),
        avg_delay=('delay_minutes', 'mean'),
        avg_avoided=('avoided_delay_minutes', 'mean'),
    ).reset_index().sort_values('week')
    model_log['model_accuracy_pct'] = np.clip(82 + np.arange(len(model_log)) * 1.7 + np.random.normal(0, 1.2, len(model_log)), 78, 97).round(1)
    model_log['rule_updates'] = np.random.randint(2, 9, len(model_log))
    model_log['milestone'] = model_log.apply(lambda r: f"Week {int(r['week'])}: model accuracy {r['model_accuracy_pct']}%, {int(r['rule_updates'])} rules updated", axis=1)
    numeric = ['ai_coverage_pct', 'avg_delay', 'avg_avoided']
    model_log[numeric] = model_log[numeric].round(1)
    return model_log


model_log_df = build_model_improvement_log(history_df)
model_log_df.tail(10)

## 10. Analytics Insights and Recommendations

In [ ]:
def build_analytics_recommendations(kpis, causes, heatmap, resources, model_log):
    recs = []
    top_cause = causes.iloc[0]
    recs.append({
        'priority_rank': 1,
        'type': 'delay_cause',
        'severity': 'High',
        'message': f"{top_cause['delay_cause']} is the largest delay contributor with {top_cause['total_delay_min']} minutes.",
        'recommendation': 'Add targeted mitigation rule and monitor impact next 7 days.',
    })

    peak_day = heatmap.sum(axis=1).idxmax()
    peak_hour = int(heatmap.sum(axis=0).idxmax())
    recs.append({
        'priority_rank': 2,
        'type': 'delay_heatmap',
        'severity': 'Medium',
        'message': f"Peak delay pattern occurs on {peak_day} around {peak_hour}:00.",
        'recommendation': 'Pre-position crew, equipment, and gate buffers before this demand wave.',
    })

    overloaded = resources.sort_values('avg_utilization_pct', ascending=False).head(1).iloc[0]
    recs.append({
        'priority_rank': 2,
        'type': 'resource_utilization',
        'severity': 'Medium' if overloaded['avg_utilization_pct'] < 90 else 'High',
        'message': f"{overloaded['resource_type']} average utilization is {overloaded['avg_utilization_pct']}%.",
        'recommendation': 'Balance demand using adjacent terminal capacity or reserve resource pool.',
    })

    latest_model = model_log.sort_values('week').iloc[-1]
    recs.append({
        'priority_rank': 3,
        'type': 'model_improvement',
        'severity': 'Low',
        'message': latest_model['milestone'],
        'recommendation': 'Promote latest model if shadow evaluation remains stable for 48 hours.',
    })

    if kpis.iloc[0]['turnaround_improvement_min'] < 4:
        recs.append({
            'priority_rank': 2,
            'type': 'turnaround_trend',
            'severity': 'Medium',
            'message': 'Turnaround improvement is below target threshold.',
            'recommendation': 'Review parallel task sequencing for cleaning, fueling, baggage, and boarding.',
        })

    return pd.DataFrame(recs).sort_values(['priority_rank', 'severity']).reset_index(drop=True)


recommendations_df = build_analytics_recommendations(kpi_df, delay_causes_df, delay_heatmap_df, resource_summary_df, model_log_df)
recommendations_df

## 11. Period Comparison

In [ ]:
def compare_periods(current, previous):
    rows = []
    for col in current.columns:
        cur = current.iloc[0][col]
        prev = previous.iloc[0][col] if col in previous.columns else np.nan
        if isinstance(cur, (int, float, np.integer, np.floating)) and pd.notna(prev):
            delta = cur - prev
            pct = (delta / prev * 100) if prev else np.nan
            rows.append({'metric': col, 'current': cur, 'previous': prev, 'delta': round(delta, 1), 'delta_pct': round(pct, 1) if pd.notna(pct) else np.nan})
    return pd.DataFrame(rows)


period_comparison_df = compare_periods(kpi_df, previous_kpi_df)
period_comparison_df

## 12. Analytics Visuals

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

recent_daily = daily_trends_df.tail(14)
axes[0, 0].plot(pd.to_datetime(recent_daily['date']), recent_daily['avg_turnaround_min'], marker='o', color='#0ea5e9')
axes[0, 0].set_title('Average Turnaround Trend')
axes[0, 0].tick_params(axis='x', rotation=35)
axes[0, 0].set_ylabel('Minutes')

top_causes = delay_causes_df.head(6)
axes[0, 1].barh(top_causes['delay_cause'], top_causes['total_delay_min'], color='#7c3aed')
axes[0, 1].set_title('Top Delay Causes')
axes[0, 1].set_xlabel('Delay minutes')

heat = delay_heatmap_df.iloc[:, 6:22]
im = axes[1, 0].imshow(heat.values, aspect='auto', cmap='Purples')
axes[1, 0].set_title('Delay Heatmap (6-21h)')
axes[1, 0].set_yticks(range(len(heat.index)))
axes[1, 0].set_yticklabels(heat.index)
axes[1, 0].set_xticks(range(len(heat.columns)))
axes[1, 0].set_xticklabels(heat.columns, rotation=45)
plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

axes[1, 1].barh(resource_summary_df['resource_type'], resource_summary_df['avg_utilization_pct'], color='#059669')
axes[1, 1].axvline(85, color='#f59e0b', linestyle='--', label='Tight')
axes[1, 1].set_title('Resource Utilization')
axes[1, 1].set_xlabel('Utilization %')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 13. Backend-Ready Payload

In [ ]:
def build_analytics_payload(kpis, comparison, daily, causes, heatmap, resources, model_log, recommendations):
    return {
        'generated_at': now.isoformat(),
        'kpis': kpis.iloc[0].to_dict(),
        'period_comparison': comparison.to_dict(orient='records'),
        'turnaround_trend': daily.tail(30).assign(date=lambda df: df['date'].astype(str)).to_dict(orient='records'),
        'top_delay_causes': causes.to_dict(orient='records'),
        'delay_heatmap': {
            'days': heatmap.index.tolist(),
            'hours': heatmap.columns.tolist(),
            'matrix': heatmap.values.tolist(),
        },
        'resource_utilization_history': resources.to_dict(orient='records'),
        'model_improvement_log': model_log.tail(12).to_dict(orient='records'),
        'recommendations': recommendations.to_dict(orient='records'),
    }


payload = build_analytics_payload(kpi_df, period_comparison_df, daily_trends_df, delay_causes_df, delay_heatmap_df, resource_summary_df, model_log_df, recommendations_df)
payload

In [ ]:
print('ANALYTICS SUMMARY')
print('=================')
print(f"Flights analyzed: {payload['kpis']['flights_analyzed']}")
print(f"Fuel cost saved: Rs {payload['kpis']['fuel_cost_saved']:,}")
print(f"Delay penalty avoided: Rs {payload['kpis']['delay_penalty_avoided']:,}")
print(f"Average turnaround: {payload['kpis']['avg_turnaround_min']} min")
print(f"Delay minutes avoided: {payload['kpis']['delay_minutes_avoided']:,}")
print(f"AI coverage: {payload['kpis']['ai_coverage_pct']}%")
print('\nTop analytics recommendations:')
for item in payload['recommendations'][:5]:
    print(f"- [{item['type']}] {item['message']} {item['recommendation']}")